Due to a typo on the mappings of satellite glial cells, its markers will need a re-do on our end. Oops. 

In [55]:
import pandas as pd 
from wikidata2df import wikidata2df

gene_reference = pd.read_csv("../results/human_gene_reference_from_panglao_to_wikidata_04_11_2020.csv")

cell_type_reference = pd.read_csv("../results/cell_type_reference_from_panglao_to_wikidata_31_10_2020.csv")

markers = pd.read_csv("../data/PanglaoDB_markers_27_Mar_2020.tsv", sep="\t")

query = """
SELECT ?item ?itemLabel ?superclass
WHERE
{
?item wdt:P31 wd:Q189118. 
?item wdt:P279 ?superclass. 
?superclass  wdt:P31 wd:Q189118. # cell type
?item wdt:P703 wd:Q15978631. # human cell type
SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
"""

dataframe_to_join = wikidata2df(query)

cell_type_reference = cell_type_reference.merge(dataframe_to_join, left_on="wikidata", right_on="superclass")

In [5]:
markers = markers.dropna(subset=["species"])
human_markers = markers[["Hs" in val for val in markers["species"]]]
human_markers_lean = human_markers[["official gene symbol", "cell type"]]

human_markers_lean = human_markers_lean.merge(cell_type_reference, left_on="cell type", right_on="panglao")[["official gene symbol", "cell type", "item"]]

human_markers_lean.columns = ["official gene symbol", "cell type", "cell type id"]
human_markers_lean = human_markers_lean.merge(gene_reference, left_on="official gene symbol", right_on="panglao")[["official gene symbol", "cell type", "cell type id", "wikidata"]]

human_markers_lean.columns = ["official gene symbol", "cell type (general)", "cell type id (human)", "gene id"]
human_markers_lean = human_markers_lean.drop_duplicates()
human_markers_lean

,official gene symbol,cell type (general),cell type id (human),gene id
0,CTRB1,Acinar cells,Q101405107,Q17911638
1,CTRB1,Enterocytes,Q101405073,Q17911638
2,KLK1,Acinar cells,Q101405107,Q18028299
3,KLK1,Distal tubule cells,Q101405088,Q18028299
4,KLK1,Endothelial cells,Q68621315,Q18028299
...,...,...,...,...
10488,SLC14A1,Urothelial cells,Q101404883,Q18031563
10489,UPK3A,Urothelial cells,Q101404883,Q18032295
10490,UPK1A,Urothelial cells,Q101404883,Q18036055
10491,UPK2,Urothelial cells,Q101404883,Q18032294


In [22]:
from wikibaseintegrator import wbi_login, WikibaseIntegrator
from wikibaseintegrator.wbi_enums import ActionIfExists
from wikibaseintegrator.wbi_config import config as wbi_config
from getpass import getpass

wbi_config['USER_AGENT'] = 'MyWikibaseBot/1.0 (https://www.wikidata.org/wiki/User:MyUsername)'

WBUSER = getpass(prompt="username:")  
WBPASS = getpass(prompt='Enter your password: ')  
login_instance = wbi_login.Clientlogin(user=WBUSER, password=WBPASS)




In [23]:
wbi = WikibaseIntegrator(login=login_instance)


In [30]:
from wikibaseintegrator.datatypes import Item, URL, Time
from wikibaseintegrator.wbi_enums import WikibaseDatePrecision

data = []
for i, row in human_markers_lean.iterrows():
    if row["cell type (general)"] == "Satellite glial cells":
      item = wbi.item.get(entity_id=row["cell type id (human)"])
      s = row["cell type id (human)"]
      p = "P8872"
      o = row["gene id"]
      r1 = "P813"
      or1 = "+2020-11-27T00:00:00Z"
      r2 = "P854"
      or2 = 'https://panglaodb.se/markers.html'
      r3 = "P248"
      or3 = "Q99936939"


      references = [
        [
            Time(time=or1, prop_nr=r1, precision=WikibaseDatePrecision.DAY),
                     URL(value=or2, prop_nr=r2),
                     Item(value=or3, prop_nr=r3)
        ]
    ]

      marker_statement = Item(value= o, prop_nr=p, references=references)
      data.append(marker_statement)


item.claims.add(data)
item.write()

<ItemEntity @c1a250 _BaseEntity__api=<wikibaseintegrator.wikibaseintegrator.WikibaseIntegrator object at 0x7fb49601c910>
	 _BaseEntity__title='Q101404951'
	 _BaseEntity__pageid=98726744
	 _BaseEntity__lastrevid=2121472022
	 _BaseEntity__type='item'
	 _BaseEntity__id='Q101404951'
	 _BaseEntity__claims=<Claims @1c130 _Claims__claims={'P703': [<Item @f08820 _Claim__mainsnak=<Snak @2ab50 _Snak__snaktype=<WikibaseSnakType.KNOWN_VALUE: 'value'> _Snak__property_number='P703' _Snak__hash='5aed5809d737aec10f4ff5eb20d35ccec83d0cb5' _Snak__datavalue={'value': {'entity-type': 'item', 'numeric-id': 15978631, 'id': 'Q15978631'}, 'type': 'wikibase-entityid'} _Snak__datatype='wikibase-item'> _Claim__type='statement' _Claim__qualifiers=<Qualifiers @2a6d0 _Qualifiers__qualifiers={}> _Claim__qualifiers_order=[] _Claim__id='Q101404951$13A3253A-7BD2-4A77-BAB9-012E5671991D' _Claim__rank=<WikibaseRank.NORMAL: 'normal'> _Claim__removed=False _Claim__references=<References @2a730 _References__references=[]>>],

In [59]:
import pandas as pd 
from wikidata2df import wikidata2df

gene_reference = pd.read_csv("../results/human_gene_reference_from_panglao_to_wikidata_04_11_2020.csv")

cell_type_reference = pd.read_csv("../results/cell_type_reference_from_panglao_to_wikidata_31_10_2020.csv")

markers = pd.read_csv("../data/PanglaoDB_markers_27_Mar_2020.tsv", sep="\t")

query = """
SELECT ?item ?itemLabel ?superclass
WHERE
{
?item wdt:P31 wd:Q189118. 
?item wdt:P279 ?superclass. 
?superclass  wdt:P31 wd:Q189118. # cell type
?item wdt:P703 wd:Q83310. # human cell type
SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
"""

dataframe_to_join = wikidata2df(query)

cell_type_reference = cell_type_reference.merge(dataframe_to_join, left_on="wikidata", right_on="superclass")

markers = markers.dropna(subset=["species"])  
mouse_markers = markers[["Mm" in val for val in markers["species"]]]
mouse_markers_lean = mouse_markers[["official gene symbol", "cell type"]]
mouse_markers_lean = mouse_markers_lean.merge(cell_type_reference, left_on="cell type", right_on="panglao")[["official gene symbol", "cell type", "item"]]
mouse_markers_lean.columns = ["official gene symbol", "cell type", "cell type id"]

mouse_gene_reference = pd.read_csv("../results/mouse_gene_reference_from_panglao_to_wikidata_22_12_2020.csv")

mouse_markers_lean = mouse_markers_lean.merge(mouse_gene_reference, left_on="official gene symbol", right_on="official gene symbol")

mouse_markers_lean

,official gene symbol,cell type,cell type id,item
0,CTRB1,Acinar cells,Q104416307,Q18261856
1,CTRB1,Acinar cells,Q104416307,Q18261856
2,CTRB1,Enterocytes,Q104416276,Q18261856
3,CTRB1,Enterocytes,Q104416276,Q18261856
4,KLK1,Acinar cells,Q104416307,Q18251870
...,...,...,...,...
34757,SLC14A1,Urothelial cells,Q104416156,Q18272870
34758,UPK3A,Urothelial cells,Q104416156,Q18255259
34759,UPK1A,Urothelial cells,Q104416156,Q18273793
34760,UPK2,Urothelial cells,Q104416156,Q18255258


In [66]:
from tqdm import tqdm  # Import tqdm for the progress bar functionality

cell_types_to_process = ["Acinar cells", "Satellite glial cells", "Clara cells"]
# Group data by 'cell type'
grouped_data = mouse_markers_lean.groupby('cell type')

for cell_type, group in tqdm(grouped_data, desc="Initializing..."):
    tqdm.write(f"Processing Cell Type: {cell_type}")  # Print current cell type being processed
    
    data = []  # Reset data for each cell type
    for i, row in group.iterrows():
        p = "P8872"
        o = row["item"]
        r1 = "P813"
        or1 = "+2020-11-27T00:00:00Z"
        r2 = "P854"
        or2 = 'https://panglaodb.se/markers.html'
        r3 = "P248"
        or3 = "Q99936939"

        references = [
            [
                Time(time=or1, prop_nr=r1, precision=WikibaseDatePrecision.DAY),
                URL(value=or2, prop_nr=r2),
                Item(value=or3, prop_nr=r3)
            ]
        ]

        marker_statement = Item(value=o, prop_nr=p, references=references)
        data.append(marker_statement)

    # Now, outside the loop for the current cell type group
    if data:
        # Assuming `item` is already the correct Wikidata item for the current cell type
        item = wbi.item.get(entity_id=row["cell type id"])
        item.claims.add(data, ActionIfExists.APPEND_OR_REPLACE)
        item.write()



Initializing...:   0%|          | 0/163 [00:00<?, ?it/s]

Processing Cell Type: Acinar cells


Initializing...:   1%|          | 1/163 [00:01<03:34,  1.32s/it]

Processing Cell Type: Adipocyte progenitor cells


Initializing...:   1%|          | 2/163 [00:02<02:39,  1.01it/s]

...

Processing Cell Type: Neutrophils


Initializing...:  62%|██████▏   | 101/163 [01:46<01:43,  1.67s/it]

Processing Cell Type: Noradrenergic neurons


Initializing...:  62%|██████▏   | 101/163 [01:46<01:05,  1.06s/it]


KeyboardInterrupt: 